In [1]:
import pandas as pd
import numpy as np

In [3]:
XZ = pd.read_csv('master16')
YZ = pd.read_csv('slave16')

In [11]:
def errorcount(view, h, l):
    expected, dead, hot = np.zeros(10), np.zeros(10), np.zeros(10)
    z = pd.Series([h*j for j in range(10)], index = view.columns)
    for _,row in view.iterrows():
        y = row[np.isfinite(row)]
        x = z[np.isfinite(row)]
        N = x.size
        if N>2:
            cov = (x*y).mean() - x.mean()*y.mean()
            m = cov/x.var(ddof=0)
            q = y.mean() - m*x.mean()
            row_fit = m*z+np.full(10, q)
            exp = (row_fit>0) & (row_fit<l)
            hit = (row>0) & (row<l)
            d = (exp==1) & (hit==0)
            h = (exp==0) & (hit==1)
            expected+=exp
            dead+=d
            hot+=h
    return expected, dead, hot

In [13]:
h, l = 7, 40
exp_x, dead_x, hot_x = errorcount(XZ, h, l)
exp_y, dead_y, hot_y = errorcount(YZ, h, l)

In [27]:
N_x, n_x, = exp_x.sum(), dead_x.sum()
N_y, n_y, = exp_y.sum(), dead_y.sum()
eff_x = (N_x - n_x)/N_x
err_x = np.sqrt(eff_x*(1-eff_x)/N_x)
eff_y = (N_y - n_y)/N_y
err_y = np.sqrt(eff_y*(1-eff_y)/N_y)

In [33]:
print('eff_x = {:.04f} +/- {:.04f}'.format(eff_x,err_x))
print('eff_y = {:.04f} +/- {:.04f}'.format(eff_y,err_y))

eff_x = 0.8426 +/- 0.0003
eff_y = 0.7824 +/- 0.0004


### Verifica

In [56]:
test = XZ.iloc[3]
z = pd.Series([h*j for j in range(10)], index = XZ.columns)
y = test[np.isfinite(test)]
x = z[np.isfinite(test)]
N = x.size
if N>2:
    cov = (x*y).mean() - x.mean()*y.mean()
    m = cov/x.var(ddof=0)
    q = y.mean() - m*x.mean()
    test_fit = m*z+np.full(10, q)
    trj = (test_fit>0) & (test_fit<l)
    hit = (test>0) & (test<l)
    dead = (trj==1) & (hit==0)
    hot = (trj==0) & (hit==1)

In [58]:
test

x0          NaN
x1    30.000000
x2    32.666667
x3    38.000000
x4    34.000000
x5    10.000000
x6     4.000000
x7     2.000000
x8          NaN
x9    10.000000
Name: 3, dtype: float64

In [60]:
test_fit

x0    40.461153
x1    36.055138
x2    31.649123
x3    27.243108
x4    22.837093
x5    18.431078
x6    14.025063
x7     9.619048
x8     5.213033
x9     0.807018
dtype: float64

In [62]:
trj

x0    False
x1     True
x2     True
x3     True
x4     True
x5     True
x6     True
x7     True
x8     True
x9     True
dtype: bool

In [64]:
hit

x0    False
x1     True
x2     True
x3     True
x4     True
x5     True
x6     True
x7     True
x8    False
x9     True
Name: 3, dtype: bool

In [66]:
dead

x0    False
x1    False
x2    False
x3    False
x4    False
x5    False
x6    False
x7    False
x8     True
x9    False
dtype: bool